# Step 3.3 — Camera Multi-Frame Tracker, Global 3D Nearest-Neighbor (Upgraded) ✅

| | |
|---|---|
| **Input** | `output/step_2/yolo_global/<sample>/<camera>.json` (Step 2.3.1 — has `global_x/y/z`, `has_3d_position`) |
| **Outputs** | `output/step_3/camera/track_<id>.json` — one file per track, trajectory across ALL cameras |
| | `output/step_3/camera_tracking_summary.csv` |
| **Used by** | Step 5 (TTC estimation, camera-only baseline), Step 4 (fusion) |

---

### Development history (kept for context)
Your notebook shows a genuinely good debugging process: the first attempt only linked detections within a single sample (no cross-time trajectory), which you correctly diagnosed and fixed in a second version that builds full trajectories. That second version is the starting point here.

### Bugs fixed in that second version

1. **Field mismatch with Step 2.3.1's actual output** — expected `global_xyz` (list) and `class`, but Step 2.3.1 outputs `global_x`, `global_y`, `global_z` (separate keys) and `class_name`. Also added the missing check for `has_3d_position` — detections without a valid LiDAR match are now correctly skipped instead of causing a crash or a bad position.
2. **Within-frame double-assignment** — a track could be extended twice in the same frame since matching happened detection-by-detection with immediate updates. Fixed with one Hungarian assignment per frame, same pattern as Step 3.1/3.2.
3. **No track eviction** — the final version had dropped the `MAX_MISSED_FRAMES` logic your first draft had. Restored.

### Design change

**Unified tracking across all 6 cameras** instead of 6 independent per-camera trackers. Since Step 2.3.1 already puts every detection in the same global frame, an object crossing from one camera's view into another's should be one continuous track — not two broken ones. This also matches the LiDAR/Radar tracker pattern for consistency across your codebase.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

import shutil
from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP0_DIR, STEP2_DIR, STEP3_DIR

YOLO_GLOBAL_DIR = STEP2_DIR / "yolo_global"
CAMERA_OUT_DIR  = STEP3_DIR / "camera"
if CAMERA_OUT_DIR.exists():
    shutil.rmtree(CAMERA_OUT_DIR)   # FIXED: clear stale per-object files before this run (was: accumulated silently across re-runs, never cleared)
CAMERA_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not YOLO_GLOBAL_DIR.exists():
    raise FileNotFoundError(f"Step 2.3.1 output not found at {YOLO_GLOBAL_DIR} — run Step 2.3.1 first.")

print(f"✅ YOLO_GLOBAL_DIR: {YOLO_GLOBAL_DIR}")
print(f"✅ CAMERA_OUT_DIR : {CAMERA_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output


✅ YOLO_GLOBAL_DIR: F:\Sensor fusion Research\output\step_2\yolo_global
✅ CAMERA_OUT_DIR : F:\Sensor fusion Research\output\step_3\camera


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Named constants
# ─────────────────────────────────────────────────────────────────

DIST_THRESHOLD      = 2.0   # metres — kept at your original value (camera-derived 3D is noisier than LiDAR's own)
MAX_MISSED_FRAMES   = 3     # frames

CAMERA_NAMES = ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
                'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']

print(f"✅ DIST_THRESHOLD = {DIST_THRESHOLD}m, MAX_MISSED_FRAMES = {MAX_MISSED_FRAMES}")

✅ DIST_THRESHOLD = 2.0m, MAX_MISSED_FRAMES = 3


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Class-aware global tracker
# Same Hungarian-assignment + eviction pattern as Step 3.1/3.2,
# extended with class-name gating (a car should never match a pedestrian track)
# FIXED: scene-boundary isolation — tracks never associate across a scene_name
#        change; at a boundary all active tracks are force-finalized and a
#        fresh track table starts for the new scene (no carried IDs/state).
# ─────────────────────────────────────────────────────────────────

import uuid
import numpy as np
from scipy.optimize import linear_sum_assignment


class ClassAwareGlobalTracker:
    def __init__(self, dist_thresh, max_missed):
        self.active_tracks = {}     # tid -> {"class_name":.., "trajectory":[...], "missed": int}
        self.finished_tracks = {}
        self.dist_thresh = dist_thresh
        self.max_missed = max_missed
        self.current_scene_name = None
        self.association_log = []  # (prev_sample_id, prev_scene, curr_sample_id, curr_scene) per accepted match

    def update(self, detections, sample_id, scene_name):
        """detections: list of dicts with keys pos:[x,y,z], class_name, confidence, camera."""
        if self.current_scene_name is not None and scene_name != self.current_scene_name:
            # Scene boundary: close out every active track from the previous scene and
            # start a fresh, empty track table. No track IDs, motion state, or "last known
            # position" carry across — each survivor is just finalized as-is.
            self.finished_tracks.update(self.active_tracks)
            self.active_tracks = {}
        self.current_scene_name = scene_name

        track_ids = list(self.active_tracks.keys())
        n_tracks, n_dets = len(track_ids), len(detections)

        matched_track_idx, matched_det_idx = set(), set()

        if n_tracks > 0 and n_dets > 0:
            SENTINEL = 1e6
            cost = np.full((n_tracks, n_dets), SENTINEL)

            for i, tid in enumerate(track_ids):
                track = self.active_tracks[tid]
                traj = track["trajectory"]
                last_pos = np.array(traj[-1]["pos"], dtype=float)
                pred = last_pos
                if len(traj) >= 2:
                    vel = (last_pos - np.array(traj[-2]["pos"], dtype=float)) / 0.5   # 2 Hz keyframes
                    spd = np.linalg.norm(vel)
                    if spd > 30.0:
                        vel = vel / spd * 30.0
                    pred = last_pos + vel * 0.5
                for j, det in enumerate(detections):
                    if det["class_name"] != track["class_name"]:
                        continue  # never match across different object classes
                    d = np.linalg.norm(np.array(det["pos"]) - pred)
                    if d < self.dist_thresh:
                        cost[i, j] = d

            row_idx, col_idx = linear_sum_assignment(cost)
            for r, c in zip(row_idx, col_idx):
                if cost[r, c] < SENTINEL:   # a real match, not the "no valid pairing" sentinel
                    tid = track_ids[r]
                    det = detections[c]
                    prev_sample_id = self.active_tracks[tid]["trajectory"][-1]["sample_id"]
                    self.association_log.append((prev_sample_id, self.current_scene_name, sample_id, scene_name))
                    self.active_tracks[tid]["trajectory"].append({
                        "sample_id": sample_id, "pos": det["pos"],
                        "confidence": det["confidence"], "camera": det["camera"]
                    })
                    self.active_tracks[tid]["missed"] = 0
                    matched_track_idx.add(r)
                    matched_det_idx.add(c)

        for i, tid in enumerate(track_ids):
            if i not in matched_track_idx:
                self.active_tracks[tid]["missed"] += 1
                if self.active_tracks[tid]["missed"] > self.max_missed:
                    self.finished_tracks[tid] = self.active_tracks.pop(tid)

        for j in range(n_dets):
            if j not in matched_det_idx:
                det = detections[j]
                tid = f"cam_{uuid.uuid4().hex[:8]}"
                self.active_tracks[tid] = {
                    "class_name": det["class_name"],
                    "trajectory": [{
                        "sample_id": sample_id, "pos": det["pos"],
                        "confidence": det["confidence"], "camera": det["camera"]
                    }],
                    "missed": 0
                }

    def save_tracks(self, out_dir):
        all_tracks = {**self.finished_tracks, **self.active_tracks}
        n_saved = 0
        for tid, track in all_tracks.items():
            if len(track["trajectory"]) >= 2:
                with open(out_dir / f"track_{tid}.json", "w") as f:
                    json.dump({
                        "track_id": tid,
                        "class_name": track["class_name"],
                        "trajectory": track["trajectory"]
                    }, f, indent=2)
                n_saved += 1
        return n_saved, len(all_tracks)


print("ClassAwareGlobalTracker defined.")


ClassAwareGlobalTracker defined.


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Main loop: merge all 6 cameras per sample, then track globally
# ─────────────────────────────────────────────────────────────────

import json
from tqdm import tqdm

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

tracker = ClassAwareGlobalTracker(dist_thresh=DIST_THRESHOLD, max_missed=MAX_MISSED_FRAMES)
n_samples_processed = 0
n_detections_skipped_no_3d = 0
n_detections_used = 0

for sample_id in tqdm(sorted(samples_index.keys()), desc="Tracking camera objects"):
    sample_dir = YOLO_GLOBAL_DIR / sample_id
    if not sample_dir.exists():
        continue

    combined_detections = []

    for cam in CAMERA_NAMES:
        cam_file = sample_dir / f"{cam}.json"
        if not cam_file.exists():
            continue

        with open(cam_file) as f:
            dets = json.load(f)

        for det in dets:
            if not det.get("has_3d_position", False):   # FIXED — skip detections with no LiDAR match
                n_detections_skipped_no_3d += 1
                continue

            combined_detections.append({
                "pos": [det["global_x"], det["global_y"], det["global_z"]],   # FIXED — correct field names
                "class_name": det["class_name"],                              # FIXED — was "class"
                "confidence": det["confidence"],
                "camera": cam
            })
            n_detections_used += 1

    scene_name = samples_index[sample_id]["scene_name"]
    tracker.update(combined_detections, sample_id, scene_name)
    n_samples_processed += 1

n_saved, n_total = tracker.save_tracks(CAMERA_OUT_DIR)

# ── Scene-boundary isolation check ─────────────────────────────────
cross_scene_associations = [a for a in tracker.association_log if a[1] != a[3]]
assert len(cross_scene_associations) == 0, \
    f"{len(cross_scene_associations)} cross-scene associations found: {cross_scene_associations[:5]}"

print(f"\nStep 3.3 complete.")
print(f"   Samples processed          : {n_samples_processed}")
print(f"   Detections used (has 3D)   : {n_detections_used}")
print(f"   Detections skipped (no 3D) : {n_detections_skipped_no_3d}")
print(f"   Total tracks created       : {n_total}")
print(f"   Tracks saved (length >= 2)  : {n_saved}")
print(f"   Associations logged        : {len(tracker.association_log)} (0 cross-scene, verified)")
print(f"Saved to: {CAMERA_OUT_DIR}")


Tracking camera objects:   0%|          | 0/404 [00:00<?, ?it/s]

Tracking camera objects:   1%|          | 5/404 [00:00<00:08, 48.14it/s]

Tracking camera objects:   3%|▎         | 11/404 [00:00<00:07, 49.72it/s]

Tracking camera objects:   4%|▍         | 16/404 [00:00<00:08, 46.82it/s]

Tracking camera objects:   5%|▌         | 21/404 [00:00<00:09, 40.39it/s]

...

Tracking camera objects:  96%|█████████▌| 386/404 [00:10<00:00, 46.28it/s]

Tracking camera objects:  97%|█████████▋| 393/404 [00:10<00:00, 50.37it/s]

Tracking camera objects:  99%|█████████▉| 399/404 [00:10<00:00, 50.93it/s]

Tracking camera objects: 100%|██████████| 404/404 [00:10<00:00, 38.32it/s]


Step 3.3 complete.
   Samples processed          : 404
   Detections used (has 3D)   : 5867
   Detections skipped (no 3D) : 178
   Total tracks created       : 2599
   Tracks saved (length >= 2)  : 1131
   Associations logged        : 3268 (0 cross-scene, verified)
Saved to: F:\Sensor fusion Research\output\step_3\camera


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Track length distribution + cross-camera continuity check
# ─────────────────────────────────────────────────────────────────

import pandas as pd

track_lengths = []
cross_camera_tracks = 0

for track_file in CAMERA_OUT_DIR.glob("track_*.json"):
    with open(track_file) as f:
        track = json.load(f)
    track_lengths.append(len(track["trajectory"]))

    cams_seen = {pt["camera"] for pt in track["trajectory"]}
    if len(cams_seen) > 1:
        cross_camera_tracks += 1

length_df = pd.DataFrame({"track_length": track_lengths})
summary_path = STEP3_DIR / "camera_tracking_summary.csv"
length_df.to_csv(summary_path, index=False)

print(f"✅ Summary saved: {summary_path}")
print(f"   Total tracks         : {len(length_df)}")
print(f"   Mean track length    : {length_df['track_length'].mean():.1f} frames")
print(f"   Tracks of length 2   : {(length_df['track_length'] == 2).sum()} "
      f"({(length_df['track_length'] == 2).mean()*100:.1f}%)")
print(f"   Tracks of length 10+ : {(length_df['track_length'] >= 10).sum()}")
print(f"\n   Tracks spanning MORE THAN ONE camera: {cross_camera_tracks} "
      f"({cross_camera_tracks/len(length_df)*100:.1f}% of all tracks)")
print("   This number is only possible with the unified global tracker —")
print("   the original per-camera design could never produce it.")

display(length_df.describe())

✅ Summary saved: F:\Sensor fusion Research\output\step_3\camera_tracking_summary.csv
   Total tracks         : 1131
   Mean track length    : 3.9 frames
   Tracks of length 2   : 497 (43.9%)
   Tracks of length 10+ : 52

   Tracks spanning MORE THAN ONE camera: 522 (46.2% of all tracks)
   This number is only possible with the unified global tracker —
   the original per-camera design could never produce it.


,track_length
count,1131.000000
mean,3.889478
std,2.999880
min,2.000000
25%,2.000000
50%,3.000000
75%,5.000000
max,27.000000
